In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from astropy import units as u
from dusty_colors.postrun.dust_extinction_fit import DEFAULT_FILTER_WAVELENGTHS_UM
from dust_extinction.averages import G03_SMCBar
from dust_extinction.parameter_averages import G23
from pathlib import Path
from scipy.optimize import least_squares

from dusty_colors import use_matplotlib_style

use_matplotlib_style()

In [ ]:
stack = "dp1_default"
save_figs = True

In [ ]:
ROOT = Path().resolve().parents[0]
STACK_DIR = ROOT / "results" / "stacks" / stack
STACK_PATH = STACK_DIR / "stack_fcolors.npz"

data = np.load(STACK_PATH)

fg_redshift = 0.36
waves = {
    band: DEFAULT_FILTER_WAVELENGTHS_UM[band] / (1 + fg_redshift) for band in "griz"
}
x_rest = np.array([waves[band] for band in "giz"])
xr_rest = waves["r"]

rnorm_kpc = 20

In [ ]:
def get_points_for_bins(bins):
    """Return g-r, i-r, and z-r points with jackknife-propagated errors."""
    y = []
    yerr = []
    for i in bins:
        # Get averages
        g_r = float(data["g-r_avg"][i])
        r_i = float(data["r-i_avg"][i])
        i_z = float(data["i-z_avg"][i])
        r_z = r_i + i_z
        y.append(np.array([g_r, -r_i, -r_z]))

        # Get jackknife samples
        g_r_samples = data["g-r_jackknife_samples"][:, i]
        r_i_samples = data["r-i_jackknife_samples"][:, i]
        i_z_samples = data["i-z_jackknife_samples"][:, i]
        r_z_samples = r_i_samples + i_z_samples
        samples = np.column_stack(
            [
                g_r_samples,
                -r_i_samples,
                -r_z_samples,
            ]
        )

        # Calculate jackknife covariance
        centered = samples - samples.mean(axis=0)
        covariance = (1.0 - 1.0 / samples.shape[0]) * centered.T @ centered
        yerr.append(np.sqrt(np.diag(covariance)))

    y = np.array(y)
    yerr = np.array(yerr)

    return y, yerr


def model(x, A, index, law, bins, rnorm_kpc):
    """Predict color excess with respect to the r-band"""
    # Excess as a function of wavelength
    ratio = law((1.0 / x) * u.micron**-1)
    ratio_r = law((1.0 / xr_rest) * u.micron**-1)
    excess = ratio - ratio_r

    # Radial power law normalization
    radii = data["g-r_bin_centers"][bins]
    norm = A * (radii / rnorm_kpc) ** index

    # Rows = radial bins
    # Columns = wavelength bins
    y_model = excess[None, :] * norm[:, None]

    return y_model


def fit_powerlaw(law, bins, rnorm_kpc=20, return_result=False):
    # Get points for each bin
    y, yerr = get_points_for_bins(bins)

    # Function that returns weighted residuals
    def resid(params):
        A, index = params

        # Generate model template
        y_model = model(x_rest, A, index, law, bins, rnorm_kpc=rnorm_kpc)

        return np.ravel((y_model - y) / yerr)

    result = least_squares(resid, [0.1, -1])

    if return_result:
        return result

    if not result.success:
        raise RuntimeError("Optimization failed")

    # Extract results
    out = {
        "model_type": "powerlaw",
        "rnorm_kpc": rnorm_kpc,
        "dust_law": law,
        "param": result.x,
    }

    fisher = result.jac.T @ result.jac
    out["param_cov"] = np.linalg.pinv(fisher)
    out["param_err"] = np.sqrt(np.diag(out["param_cov"]))
    out["snr"] = np.abs(out["param"] / out["param_err"])

    out["chi2"] = np.sum(resid(result.x) ** 2)
    out["dof"] = y.size - len(out["param"])
    out["chi2/dof"] = out["chi2"] / out["dof"]

    return out


def fit_norm(law, bins, return_result=False):
    # Get points for each bin
    y, yerr = get_points_for_bins(bins)

    # Function that returns weighted residuals
    def resid(params):
        A = params

        # Generate model template
        y_model = model(x_rest, A, 0, law, bins, rnorm_kpc=100)

        return np.ravel((y_model - y) / yerr)

    result = least_squares(resid, [y.mean()])

    if return_result:
        return result

    if not result.success:
        raise RuntimeError("Optimization failed")

    # Extract results
    out = {
        "model_type": "norm",
        "dust_law": law,
        "param": result.x,
    }

    fisher = result.jac.T @ result.jac
    out["param_cov"] = np.linalg.pinv(fisher)
    out["param_err"] = np.sqrt(np.diag(out["param_cov"]))
    out["snr"] = np.abs(out["param"] / out["param_err"])

    out["chi2"] = np.sum(resid(result.x) ** 2)
    out["dof"] = y.size - len(out["param"])
    out["chi2/dof"] = out["chi2"] / out["dof"]

    return out


def evaluate_model(x, params, bins):
    if params["model_type"] == "powerlaw":
        A, index = params["param"]
        return model(x, A, index, params["dust_law"], bins, params["rnorm_kpc"])
    else:
        A, index = params["param"], 0.0
        return model(x, A, index, params["dust_law"], bins, 100)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(3, 5), dpi=150, sharex=True)

# Define dust laws we will fit
laws = {
    "SMC": {
        "model": G03_SMCBar(),
        "color": "C3",
        "ytext": 0.8,
    },
    "MW": {
        "model": G23(Rv=3.1),
        "color": "C0",
        "ytext": 0.6,
    },
}

# Loop over radial bins
x_grid = np.linspace(0.3, 0.7, 1000)
y, yerr = get_points_for_bins([0, 1, 2, 3])
for i, (ax, yi, ei) in enumerate(zip(axes, y, yerr)):
    # Plot measurements
    ax.scatter(xr_rest, 0, marker="x", c="k", zorder=10)
    ax.errorbar(x_rest, yi, ei, ls="", marker=".", capsize=2, c="k", zorder=10)

    # Fit and plot models
    for name, law in laws.items():
        # Fit and plot
        result = fit_norm(law["model"], [i])
        y_model = evaluate_model(x_grid, result, [i])
        ax.plot(x_grid, y_model[0], c=law["color"], ls="--")

        # Print chi2/dof
        text = rf"{name}: $\chi^2/\nu$ = {result["chi2/dof"]:.2f}"
        ax.text(
            0.95,
            law["ytext"],
            text,
            transform=ax.transAxes,
            color=law["color"],
            fontsize=7,
            ha="right",
        )

# Axis settings
fig.supylabel(r"Relative Extinction $A_\lambda - A_r$ [mag]", x=-0.07)
axes[-1].set_xlabel(r"$\lambda_\mathrm{rest}$ [$\mu$m]")
for ax in axes:
    ax.set(xlim=(x_grid.min(), x_grid.max()))
    ax.minorticks_on()
    ax.tick_params(axis="both", which="both", direction="in", top=True, right=True)
axes[0].set(ylim=(-0.2, 0.25))
axes[1].set(ylim=(-0.045, 0.065), yticks=np.arange(-0.04, 0.08, 0.04))

# Label bands at top of figure
for band in waves:
    axes[0].text(waves[band], 0.27, f"${band}$", ha="center", va="bottom")

xtext, ytext = 0.03, 0.1
axes[0].text(
    xtext,
    ytext,
    r"$10 < r_\perp < 15 ~ \mathrm{kpc}$",
    transform=axes[0].transAxes,
    fontsize=7,
)
axes[1].text(
    xtext,
    ytext,
    r"$15 < r_\perp < 40 ~ \mathrm{kpc}$",
    transform=axes[1].transAxes,
    fontsize=7,
)
axes[2].text(
    xtext,
    ytext,
    r"$40 < r_\perp < 120 ~ \mathrm{kpc}$",
    transform=axes[2].transAxes,
    fontsize=7,
)
axes[3].text(
    xtext,
    ytext,
    r"$120 < r_\perp < 1000 ~ \mathrm{kpc}$",
    transform=axes[3].transAxes,
    fontsize=7,
)


fig.subplots_adjust(hspace=0.1)

if save_figs:
    fig.savefig("../figures/fig_chromaticity.pdf", bbox_inches="tight")

In [ ]:
def get_shape_points(bins, results=None):
    """Return extinction relative to the r band, normalized by $A_r - A_i$.

    The requested bins are co-added, weighted by the signal-to-noise of the
    normalizing color. Dividing each jackknife sample by its own $A_r - A_i$
    cancels any overall rescaling of the extinction curve, so the returned
    errors describe only the shape of the wavelength dependence.

    Pass `results` to read a stack other than the one loaded above.
    """
    results = data if results is None else results

    def excess(g_r, r_i, i_z):
        """Stack g-r, i-r, and z-r along the last axis."""
        return np.stack([g_r, -r_i, -(r_i + i_z)], axis=-1)

    weights = [
        abs(results["r-i_avg"][i]) / np.var(results["r-i_jackknife_samples"][:, i])
        for i in bins
    ]

    point = sum(
        w * excess(results["g-r_avg"][i], results["r-i_avg"][i], results["i-z_avg"][i])
        for w, i in zip(weights, bins)
    )
    samples = sum(
        w
        * excess(
            results["g-r_jackknife_samples"][:, i],
            results["r-i_jackknife_samples"][:, i],
            results["i-z_jackknife_samples"][:, i],
        )
        for w, i in zip(weights, bins)
    )

    # Normalize every jackknife sample by its own i-r point, then propagate
    normalized = samples / -samples[:, [1]]
    centered = normalized - normalized.mean(axis=0)
    covariance = (1.0 - 1.0 / normalized.shape[0]) * centered.T @ centered

    return point / -point[1], np.sqrt(np.diag(covariance))


def normalized_model(x, law):
    """Predict extinction relative to the r band, normalized by $A_r - A_i$."""
    ratio_r = law((1.0 / xr_rest) * u.micron**-1)
    excess = law((1.0 / x) * u.micron**-1) - ratio_r
    excess_i = law((1.0 / waves["i"]) * u.micron**-1) - ratio_r

    return excess / -excess_i


fig, ax = plt.subplots(figsize=(3, 1.5), dpi=150)

# The g and z points carry the shape information; i is fixed by the
# normalization and r is zero by construction
shape_bands = [0, 2]
x_grid = np.linspace(0.3, 0.7, 1000)

# No normalization is fit here: both curves are scaled exactly like the data
for name, law in laws.items():
    ax.plot(
        x_grid,
        normalized_model(x_grid, law["model"]),
        c=law["color"],
        ls="--",
        label=name,
    )

# Individual radial bins. Bin 2 is omitted because its normalizing color has
# S/N < 1, which makes its ratio errors meaningless rather than merely large.
for i, offset in zip([0, 1, 3], [-0.008, 0.0, 0.008]):
    y, yerr = get_shape_points([i])
    ax.errorbar(
        x_rest[shape_bands] + offset,
        y[shape_bands],
        yerr[shape_bands],
        ls="",
        marker=".",
        ms=3,
        capsize=0,
        c="0.7",
        lw=0.8,
    )

# All bins combined, which is the measurement the curves should be judged against
y, yerr = get_shape_points([0, 1, 2, 3])
ax.errorbar(
    x_rest[shape_bands],
    y[shape_bands],
    yerr[shape_bands],
    ls="",
    marker=".",
    capsize=2,
    c="k",
    zorder=10,
    label="All bins",
)
ax.scatter([xr_rest, waves["i"]], [0, -1], marker="x", c="k", zorder=10)

# Axis settings
ax.set_xlabel(r"$\lambda_\mathrm{rest}$ [$\mu$m]")
ax.set_ylabel(r"$(A_\lambda - A_r) / (A_r - A_i)$")
ax.set(xlim=(x_grid.min(), x_grid.max()), ylim=(-2.6, 4.4))
ax.minorticks_on()
ax.tick_params(axis="both", which="both", direction="in", top=True, right=True)
ax.legend(fontsize=7, frameon=False, loc="upper right")

# Label bands at top of figure
for band in waves:
    ax.text(waves[band], 4.5, f"${band}$", ha="center", va="bottom")

if save_figs:
    fig.savefig("../figures/fig_chromaticity_shape.pdf", bbox_inches="tight")

In [ ]:
# Analysis variants, matching the sensitivity figure in plot_main_results.ipynb
variants = [
    ("dp1_default", "Default", "fcolors"),
    ("dp1_uniform_random", "Uniform randoms", "fcolors"),
    ("dp1_no_random", "No random correction", "fcolors"),
    ("dp1_no_flip", "No flipped correction", "fcolors"),
    ("dp1_smaller_ap", "Smaller aperture", "fcolors"),
    ("dp1_larger_ap", "Larger aperture", "fcolors"),
    ("dp1_smaller_dz", r"$\Delta z = 0.1$", "fcolors"),
    ("dp1_larger_dz", r"$\Delta z = 0.3$", "fcolors"),
    ("dp1_mcolors", "Magnitude colors", "mcolors"),
    ("dp1_no_sv38", "No SV 38 7", "fcolors"),
]

fig, axes = plt.subplots(2, 1, figsize=(4, 3), dpi=200, sharex=True)

# The g and z points are the only ones carrying shape information, so each gets
# its own panel. Variants run along x rather than being offset in wavelength,
# because the normalized curves are steep enough near g that an offset large
# enough to be legible would shift the models by about one error bar.
for ax, band in zip(axes, "gz"):
    for name, law in laws.items():
        ax.axhline(
            float(normalized_model(waves[band], law["model"])),
            c=law["color"],
            ls="--",
            label=name,
        )

    for i, (variant, label, mode) in enumerate(variants):
        results = np.load(ROOT / "results" / "stacks" / variant / f"stack_{mode}.npz")
        y, yerr = get_shape_points([0, 1, 2, 3], results)
        j = "giz".index(band)
        default = variant == "dp1_default"
        ax.errorbar(
            i,
            y[j],
            yerr[j],
            ls="",
            marker="s",
            markersize=4 if default else 3,
            mfc="w" if default else None,
            c="k",
            zorder=10,
        )

    # Limits are set so the informative scatter is legible; the much noisier
    # no-flipped-correction bars run off the panel
    ax.set(
        ylabel=rf"$(A_{band} - A_r) / (A_r - A_i)$",
        ylim={"g": (0.6, 3.6), "z": (-2.4, -1.15)}[band],
        xlim=(-0.6, len(variants) - 0.4),
    )
    ax.minorticks_on()
    ax.tick_params(axis="both", which="both", direction="in", top=True, right=True)
    ax.tick_params(axis="x", which="minor", bottom=False, top=False)

axes[0].legend(loc="upper right", fontsize=7, frameon=False, ncols=2)
axes[-1].set_xticks(
    range(len(variants)), [label for _, label, _ in variants], rotation=45, ha="right"
)
fig.subplots_adjust(hspace=0.1)

if save_figs:
    fig.savefig(
        "../figures/fig_chromaticity_shape_sensitivity.pdf", bbox_inches="tight"
    )